# 05 — Catalog Enrichment

Adds the catalog metadata that downstream tools (AI/BI, Genie, Discover) rely
on to surface the right thing automatically:

- **Comments** on every gold table and key column
- **Tags** — `domain=hockey_analytics`, `tier=bronze|silver|gold`, `data_owner`
- **Column-level tags** — `pii` flag on player birth fields (when ingested)
- A lineage check via `system.access.column_lineage` so you can see the bronze→silver→gold path

In [ ]:
import os, json
from dotenv import load_dotenv
load_dotenv()

# Dual-mode Spark session — works locally via Databricks Connect AND inside a
# Databricks workspace notebook. In the workspace, `spark` already exists.
try:
    spark
except NameError:
    from databricks.connect import DatabricksSession
    spark = DatabricksSession.builder.serverless().getOrCreate()

from databricks.sdk import WorkspaceClient
w = WorkspaceClient()

In [ ]:
UC_CATALOG    = os.getenv("UC_CATALOG", "alexander_booth")
UC_SCHEMA     = os.getenv("UC_SCHEMA", "sportlogiq_nhl")
BRONZE_SCHEMA = f"{UC_SCHEMA}_bronze"
SILVER_SCHEMA = f"{UC_SCHEMA}_silver"
GOLD_SCHEMA   = f"{UC_SCHEMA}_gold"
VOLUME_NAME   = "raw_data"
VOLUME_PATH   = f"/Volumes/{UC_CATALOG}/{BRONZE_SCHEMA}/{VOLUME_NAME}"

print(f"Catalog: {UC_CATALOG}")
print(f"Bronze:  {UC_CATALOG}.{BRONZE_SCHEMA}")
print(f"Silver:  {UC_CATALOG}.{SILVER_SCHEMA}")
print(f"Gold:    {UC_CATALOG}.{GOLD_SCHEMA}")
print(f"Volume:  {VOLUME_PATH}")

In [ ]:
B = f"{UC_CATALOG}.{BRONZE_SCHEMA}"
S = f"{UC_CATALOG}.{SILVER_SCHEMA}"
G = f"{UC_CATALOG}.{GOLD_SCHEMA}"

DATA_OWNER = os.getenv("DATA_OWNER", "hockey_analytics_team")
DOMAIN     = os.getenv("DOMAIN",     "hockey_analytics")
print(f"Domain: {DOMAIN}; data_owner: {DATA_OWNER}")

In [ ]:
def safe_set_tags(target: str, tags: dict):
    """Apply tags but tolerate governed-tag policy errors."""
    pairs = ", ".join([f"'{k}' = '{v}'" for k, v in tags.items()])
    try:
        spark.sql(f"ALTER {target} SET TAGS ({pairs})")
        print(f"  tagged {target}: {tags}")
    except Exception as e:
        print(f"  could not tag {target}: {str(e)[:120]}")


# Catalog + schema-level tags for chargeback / discovery
for schema, tier in [(BRONZE_SCHEMA, "bronze"), (SILVER_SCHEMA, "silver"), (GOLD_SCHEMA, "gold")]:
    safe_set_tags(f"SCHEMA {UC_CATALOG}.{schema}",
                  {"domain": DOMAIN, "tier": tier, "data_owner": DATA_OWNER})

In [ ]:
# Gold table comments — short, business-readable. AI/BI + Genie use these directly.
TABLE_COMMENTS = {
    "dim_team":            "NHL teams as captured by SportLogiq. Type-1 dimension.",
    "dim_player":          "Skaters and goalies as captured by SportLogiq. Type-1 dimension.",
    "dim_venue":           "NHL arenas / playing venues. Type-1 dimension.",
    "dim_game":            "One row per scheduled NHL game with venue and team foreign keys.",
    "dim_date":            "Calendar date dimension covering 2020-01-01 through today + 90d.",
    "fact_game_events":    "Compiled on-ice events (shots, hits, faceoffs, possessions). x_coord/y_coord are SportLogiq rink coordinates.",
    "fact_player_shifts":  "Per-shift events (on-ice, off-ice, line changes).",
    "fact_player_game_metrics":  "Long-form per-game metric grid: one row per (game, player, topic, metric_key).",
    "fact_player_season_metrics":"Long-form season-to-date metric grid for player/team/goalie/opposingteam scopes.",
}
for t, desc in TABLE_COMMENTS.items():
    spark.sql(f"COMMENT ON TABLE {G}.{t} IS '{desc}'")
    print(f"  commented {G}.{t}")

In [ ]:
# Column comments where Genie struggles without them — sport-specific terminology
COLUMN_COMMENTS = [
    (f"{G}.fact_game_events.x_coord",
     "Rink x-coordinate from SportLogiq (centre-ice = 0). Used for shot maps."),
    (f"{G}.fact_game_events.y_coord",
     "Rink y-coordinate from SportLogiq. Used for shot maps."),
    (f"{G}.fact_game_events.zone",
     "Zone classification: O (offensive), D (defensive), N (neutral)."),
    (f"{G}.fact_game_events.is_shot",
     "TRUE if the event is any kind of shot (on-net, blocked, missed)."),
    (f"{G}.fact_game_events.is_goal",
     "TRUE for events with outcome='goal'."),
    (f"{G}.fact_player_game_metrics.topic_id",
     "SportLogiq metric_topic id. Join to silver_metric_topics for human-readable names."),
    (f"{G}.fact_player_game_metrics.metric_key",
     "Metric short key — e.g. corsi_for, fenwick_for, expected_goals. Look up display name in metric_topics."),
]
for col, txt in COLUMN_COMMENTS:
    try:
        spark.sql(f"COMMENT ON COLUMN {col} IS '{txt}'")
    except Exception as e:
        print(f"  skipped {col}: {str(e)[:80]}")
print("column comments applied.")

In [ ]:
# Tier + domain tags on the gold tables
for t in TABLE_COMMENTS:
    safe_set_tags(f"TABLE {G}.{t}",
                  {"domain": DOMAIN, "tier": "gold", "data_owner": DATA_OWNER})

## Lineage check

Confirms the column-level lineage graph SportLogiq → bronze → silver → gold is
fully connected. Run after at least one full pipeline pass; lineage entries
require the gold tables to have been queried at least once.

In [ ]:
try:
    spark.sql(f"""
        SELECT source_table_full_name, target_table_full_name, source_column_name, target_column_name
        FROM system.access.column_lineage
        WHERE target_table_catalog = '{UC_CATALOG}'
          AND target_table_schema  = '{GOLD_SCHEMA}'
        ORDER BY target_table_full_name
        LIMIT 20
    """).show(truncate=False)
except Exception as e:
    print(f"system.access.column_lineage not available in this workspace: {e}")